# Haiku MIL Classification (nested cross-validation)

**Project Name:** Haiku

## Purpose
- Reproduces the treatment-response benchmarks of manuscript **Figure 3g–l** (Methods, "MIL classification for treatment response and clinical endpoints").
- Five frozen spatial-proteomics encoders are compared under one protocol: **Haiku(mIF)** (after the projection head), **KRONOS**, **Eva**, **VirTues (official)** and **VirTues (ours)**. Only the attention-based MIL head is trained.
- Two tasks:
  - **Melanoma** anti-PD-1 response (`Response-binary`; 75 patients, prevalence 0.33).
  - **Colorectal cancer (CRC)** response (`Outcome-binary`), pooled over the two first-line regimens FOLFIRI + bevacizumab (29 patients) and FOLFOX + bevacizumab (18 patients): 47 patients, prevalence 0.68. No treatment covariate is given to the model.
- Protocol:
  - 5 repeats × 5-fold `StratifiedGroupKFold`, grouped by **patient** (no patient on both sides of any split). For CRC the folds are stratified on outcome × regimen so that both regimens appear in every split.
  - Hyperparameters are selected **separately for each encoder** by an inner 3-fold patient-grouped CV over an 8-point grid (embedding dim {32, 128} × learning rate {5e-4, 2e-3} × instance ℓ2 normalization on/off), scored by (AUROC + AUPRC) / 2.
  - The selected configuration is refit on the outer training split with early stopping on a further 15% patient-grouped validation split.
- Evaluation: the out-of-fold predictions of all repeats are **concatenated** and AUROC / AUPRC are computed once on them. 95% CIs are patient-clustered bootstrap percentile intervals and P values are two-sided paired patient-clustered bootstrap tests against Haiku (B = 2,000).

## Notes
- `patient_id` is derived from `SAMPLE_LABEL` by stripping a trailing `_<digit>` replicate suffix (e.g. `P001_1` -> `P001`).
- Every encoder sees the same held-out acquisitions, patches and marker channels (the 81 markers shared with the KRONOS and Eva vocabularies; Methods, "Spatial-proteomics encoder baselines"). Patch embeddings are precomputed once per encoder.
- Set `RUN_TRAINING = True` to rerun the nested CV. This trains about 3,000 small MIL heads per task, a few hours on one GPU. With `RUN_TRAINING = False` the notebook reads the saved out-of-fold predictions from `OUTPUT_DIR` and recomputes all statistics and plots.
- Paths and checkpoints may be environment-specific. Run cells top-to-bottom and set your config paths first.
- Supersedes the earlier single-run 5-fold version of this notebook (one Haiku-tuned configuration shared by Haiku and VirTues, mean over folds).

In [ ]:
import sys
from pathlib import Path

# Notebook lives at <repo>/downstream/ — HAIKU_ROOT points at the repo root.
HAIKU_ROOT = Path.cwd().parent
if str(HAIKU_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(HAIKU_ROOT / 'src'))

# -------- Fill these in to point at your local copies --------
EMBEDDINGS_DIR = Path('<PATH_TO_PRECOMPUTED_EMBEDDINGS>')
SAMPLES_JSON   = HAIKU_ROOT / 'overlap_samples_final.json'
MELANOMA_CSV   = HAIKU_ROOT / 'downstream' / 'Melanoma_response-to-immunotherapy.csv'
CRC_CSV        = HAIKU_ROOT / 'downstream' / 'CRC_terminal_survival.csv'
OUTPUT_DIR     = HAIKU_ROOT / 'downstream' / 'mil_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Patch embeddings of the held-out acquisitions, one file per encoder (rows aligned
# with REGION_LABEL_FILE, which maps each patch to its acquisition).
REGION_LABEL_FILE = 'virtues_holdout_region_label.pt'
ARM_FILES = {
    'haiku_postproj':   'haikuproj_holdout_embedding.pt',
    'kronos':           'kronos_holdout_embedding.pt',
    'eva':              'eva_holdout_embedding.pt',
    'virtues_official': 'virtues_official_holdout/virtual_codex_embedding.pt',
    'virtues_ours':     'virtues_holdout_embedding.pt',
}

RUN_TRAINING = False   # True: rerun the nested CV; False: reuse saved out-of-fold predictions

from haiku import setup_notebook, seed_everything
setup_notebook(project_root=str(HAIKU_ROOT))
seed_everything(42)

In [ ]:
import os, re, json, copy, random, itertools, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42

SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ARMS = list(ARM_FILES)
REF = 'haiku_postproj'
LABEL = {'haiku_postproj': 'Haiku (mIF)', 'kronos': 'KRONOS', 'eva': 'Eva',
         'virtues_official': 'VirTues (official)', 'virtues_ours': 'VirTues (ours)'}


def load_arm_bags():
    # Group each encoder's patch embeddings into one bag per acquisition.
    ref_ids = sorted(json.load(open(SAMPLES_JSON)).keys())
    rl = torch.load(EMBEDDINGS_DIR / REGION_LABEL_FILE, weights_only=True).reshape(-1).numpy()
    arms = {}
    for name, fname in ARM_FILES.items():
        e = torch.load(EMBEDDINGS_DIR / fname, weights_only=True).float().numpy()
        assert len(e) == len(rl), (name, e.shape, rl.shape)
        bags = {}
        for i in range(len(rl)):
            bags.setdefault(ref_ids[int(rl[i])], []).append(e[i])
        arms[name] = {k: np.stack(v) for k, v in bags.items()}
        print(f'{LABEL[name]:20s} dim={e.shape[1]:5d}  bags={len(bags)}')
    return arms


if RUN_TRAINING:
    arm_bags = load_arm_bags()

In [ ]:
# ---- cohorts ----
def patient_id(sample_label):
    return re.sub(r'_\d+$', '', str(sample_label))


def first_line_regimen(treatment):
    # First-line regimen with bevacizumab: 'FOLFIRI', 'FOLFOX' or None.
    m = re.search(r'Treatment\s*1\s*:\s*(.*?)\s*(?:;\s*Treatment\s*2\s*:|$)',
                  str(treatment), flags=re.IGNORECASE | re.DOTALL)
    s = re.sub(r'[^a-z0-9]', '', (m.group(1) if m else '').lower()).replace('folfox4', 'folfox')
    if 'bevacizumab' not in s:
        return None
    if 'folfiri' in s and 'folfox' not in s:
        return 'FOLFIRI'
    if 'folfox' in s and 'folfiri' not in s:
        return 'FOLFOX'
    return None


mel = pd.read_csv(MELANOMA_CSV)
mel = mel[mel['Response-binary'].notna()].copy()
mel['pid'] = mel['SAMPLE_LABEL'].map(patient_id)
mel['label'] = mel['Response-binary'].astype(int)
mel['strat'] = mel['label']

crc = pd.read_csv(CRC_CSV)
crc['regimen'] = crc['treatment'].map(first_line_regimen)
crc = crc[crc['regimen'].notna() & crc['Outcome-binary'].notna()].copy()
crc['pid'] = crc['SAMPLE_LABEL'].map(patient_id)
crc['label'] = crc['Outcome-binary'].astype(int)
crc['strat'] = 2 * crc['label'] + (crc['regimen'] == 'FOLFOX').astype(int)   # outcome x regimen

TASKS = {'melanoma': mel, 'crc_pooled': crc}
for name, df in TASKS.items():
    pat = df.groupby('pid')['label'].first()
    print(f'{name:11s} acquisitions={len(df):3d}  patients={df.pid.nunique():3d}  '
          f'patient-level prevalence={pat.mean():.3f}')
print(crc.drop_duplicates('pid')['regimen'].value_counts().to_dict())

In [ ]:
# ---- MIL dataset, model, training (identical for every encoder) ----
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def l2_normalize(x, axis=1, eps=1e-12):
    return x / np.clip(np.linalg.norm(x, ord=2, axis=axis, keepdims=True), eps, None)


def grouped_train_val_split(ids, groups, stratify=None, test_size=0.1, seed=SEED):
    # Patient-aware train/val split: no patient appears in both parts.
    ids_arr, groups = np.asarray(ids), np.asarray(groups)
    if len(set(groups)) < 2:
        return ids, ids
    if stratify is not None and len(np.unique(stratify)) >= 2:
        n_splits = max(2, int(round(1.0 / test_size)))
        for nsp in (n_splits, 5, 3, 2):
            try:
                sgkf = StratifiedGroupKFold(n_splits=nsp, shuffle=True, random_state=seed)
                tr, va = next(sgkf.split(ids_arr, stratify, groups=groups))
                return ids_arr[tr].tolist(), ids_arr[va].tolist()
            except ValueError:
                continue
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    tr, va = next(gss.split(ids_arr, groups=groups))
    return ids_arr[tr].tolist(), ids_arr[va].tolist()


class MILBagDataset(Dataset):
    def __init__(self, emb_dict, bag_ids, labels_dict, do_l2_norm=True):
        self.ids, self.bags, self.labels = [], [], []
        for bid in bag_ids:
            if bid not in emb_dict or bid not in labels_dict:
                continue
            X = np.asarray(emb_dict[bid], dtype=np.float32)
            if do_l2_norm:
                X = l2_normalize(X, axis=1)
            self.ids.append(bid); self.bags.append(torch.from_numpy(X))
            self.labels.append(int(labels_dict[bid]))
        if not self.bags:
            raise ValueError('Empty dataset')
        self.Din = self.bags[0].shape[1]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.bags[i], torch.tensor(self.labels[i], dtype=torch.long), self.ids[i]


def pad_collate(batch):
    bags, ys, ids = zip(*batch)
    B, Nmax, D = len(bags), max(b.shape[0] for b in bags), bags[0].shape[1]
    X = torch.zeros(B, Nmax, D); Mk = torch.zeros(B, Nmax, dtype=torch.bool)
    for i, b in enumerate(bags):
        X[i, :b.shape[0]] = b; Mk[i, :b.shape[0]] = True
    return X, Mk, torch.stack(ys), list(ids)


class AttnPool(nn.Module):
    def __init__(self, d, hidden=128):
        super().__init__()
        self.V = nn.Linear(d, hidden)
        self.w = nn.Linear(hidden, 1, bias=False)

    def forward(self, H, mask):
        A = self.w(torch.tanh(self.V(H))).squeeze(-1)
        A = torch.softmax(A.masked_fill(~mask, float('-inf')), dim=1)
        return torch.einsum('bn,bnd->bd', A, H), A


class MILClassifier(nn.Module):
    def __init__(self, in_dim, embed_dim=128, dropout=0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, embed_dim), nn.GELU(), nn.Dropout(dropout), nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim), nn.GELU(), nn.LayerNorm(embed_dim))
        self.pool = AttnPool(embed_dim)
        self.head = nn.Sequential(nn.Linear(embed_dim, embed_dim), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(embed_dim, 1))

    def forward(self, X, mask):
        Z, A = self.pool(self.encoder(X), mask)
        return self.head(Z).squeeze(-1), A


def train_model(emb, labels, train_ids, val_ids, hp):
    # Positive-class weighted BCE, AdamW + cosine LR, early stopping on val (AUROC+AUPRC)/2.
    set_seed(SEED)
    tr = MILBagDataset(emb, train_ids, labels, hp['l2_norm'])
    va = MILBagDataset(emb, val_ids, labels, hp['l2_norm'])
    ltr = DataLoader(tr, batch_size=hp['batch_size'], shuffle=True, collate_fn=pad_collate)
    lva = DataLoader(va, batch_size=hp['batch_size'], shuffle=False, collate_fn=pad_collate)
    model = MILClassifier(tr.Din, hp['embed_dim'], hp['dropout']).to(DEVICE)
    y_tr = np.array([labels[i] for i in tr.ids])
    pos_weight = torch.tensor([(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)],
                              dtype=torch.float32, device=DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['wd'])
    sch = CosineAnnealingLR(opt, T_max=hp['max_epochs'], eta_min=hp['lr'] * 0.01)
    best, best_state, wait = -1, None, 0
    for _ in range(hp['max_epochs']):
        model.train()
        for X, Mk, y, _ in ltr:
            logits, _ = model(X.to(DEVICE), Mk.to(DEVICE))
            loss = crit(logits, y.to(DEVICE).float())
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
        model.eval()
        Y, P = [], []
        with torch.no_grad():
            for X, Mk, y, _ in lva:
                logits, _ = model(X.to(DEVICE), Mk.to(DEVICE))
                Y.append(y.numpy()); P.append(torch.sigmoid(logits).cpu().numpy())
        Y, P = np.concatenate(Y), np.concatenate(P)
        sc = ((roc_auc_score(Y, P) + average_precision_score(Y, P)) / 2
              if len(np.unique(Y)) > 1 else 0.0)
        if sc > best:
            best, best_state, wait = sc, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= hp['patience']:
                break
    model.load_state_dict(best_state)
    return model


@torch.no_grad()
def predict(model, emb, labels, ids, hp):
    ds = MILBagDataset(emb, ids, labels, hp['l2_norm'])
    dl = DataLoader(ds, batch_size=256, shuffle=False, collate_fn=pad_collate)
    model.eval()
    I, Y, P = [], [], []
    for X, Mk, y, bid in dl:
        logits, _ = model(X.to(DEVICE), Mk.to(DEVICE))
        I += bid; Y.append(y.numpy()); P.append(torch.sigmoid(logits).cpu().numpy())
    out = pd.DataFrame({'id': I, 'label': np.concatenate(Y), 'prob': np.concatenate(P)})
    score = ((roc_auc_score(out.label, out.prob) + average_precision_score(out.label, out.prob)) / 2
             if out.label.nunique() > 1 else np.nan)
    return score, out

In [ ]:
# ---- nested cross-validation ----
GRID_AXES = {'embed_dim': [32, 128], 'lr': [5e-4, 2e-3], 'l2_norm': [True, False]}
FIXED = {'dropout': 0.1, 'wd': 1e-4, 'batch_size': 32, 'max_epochs': 50, 'patience': 10}
GRID = [dict(FIXED, **dict(zip(GRID_AXES, v))) for v in itertools.product(*GRID_AXES.values())]
N_REPEATS, N_OUTER, N_INNER = 5, 5, 3


def inner_select(emb, labels, ids, groups, strat):
    # Pick one configuration for ONE encoder using only the outer training split.
    ids, g, s = np.asarray(ids), np.asarray(groups), np.asarray(strat)
    n_eff = min(N_INNER, len(np.unique(g)), int(np.bincount(s).min()))
    if n_eff < 2:
        return GRID[0]
    splits = list(StratifiedGroupKFold(n_splits=n_eff, shuffle=True, random_state=SEED)
                  .split(ids, s, groups=g))
    best, best_s = GRID[0], -np.inf
    for hp in GRID:
        ss = []
        for itr, iva in splits:
            try:
                m = train_model(emb, labels, ids[itr].tolist(), ids[iva].tolist(), hp)
                ss.append(predict(m, emb, labels, ids[iva].tolist(), hp)[0])
            except Exception:
                ss.append(np.nan)
        sc = np.nanmean(ss) if not np.all(np.isnan(ss)) else -np.inf
        if sc > best_s:
            best_s, best = sc, hp
    return best


def run_nested(df, arm_bags):
    ids = sorted(set(df.ACQUISITION_ID) & set.intersection(*[set(b) for b in arm_bags.values()]))
    d = df.set_index('ACQUISITION_ID').loc[ids]
    labels = d['label'].to_dict()
    y, groups, strat = d['label'].values, d['pid'].values, d['strat'].values
    rows = []
    for rep in range(N_REPEATS):
        outer = StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=1000 + rep)
        for fold, (tr, te) in enumerate(outer.split(ids, strat, groups=groups)):
            assert not set(groups[tr]) & set(groups[te]), 'patient leakage'
            tr_ids, te_ids = [ids[i] for i in tr], [ids[i] for i in te]
            for arm in ARMS:
                emb = arm_bags[arm]
                hp = inner_select(emb, labels, tr_ids, groups[tr], strat[tr])
                sub_tr, sub_va = grouped_train_val_split(tr_ids, groups[tr], stratify=y[tr],
                                                         test_size=0.15, seed=SEED)
                model = train_model(emb, labels, sub_tr, sub_va, hp)
                _, pred = predict(model, emb, labels, te_ids, hp)
                pred['arm'], pred['rep'], pred['fold'] = arm, rep, fold
                pred['embed_dim'], pred['lr'], pred['l2_norm'] = hp['embed_dim'], hp['lr'], hp['l2_norm']
                rows.append(pred)
            print(f'  repeat {rep} fold {fold} done')
    return pd.concat(rows, ignore_index=True)

In [ ]:
# ---- run (or reload) the nested CV ----
OOF = {}
for name, df in TASKS.items():
    path = OUTPUT_DIR / f'mil_cls_{name}_oof_predictions.csv'
    if RUN_TRAINING:
        print(f'== {name}')
        run_nested(df, arm_bags).to_csv(path, index=False)
    OOF[name] = pd.read_csv(path)
    o = OOF[name]
    print(f'{name:11s} {len(o[o.arm == REF])} pooled out-of-fold predictions per encoder '
          f'({o.rep.nunique()} repeats x {o.groupby(["rep", "fold"]).ngroups // o.rep.nunique()} folds)')

In [ ]:
# ---- pooled out-of-fold AUROC / AUPRC, patient-clustered bootstrap ----
N_BOOT, BOOT_SEED = 2000, 0
METRICS = {'AUROC': roc_auc_score, 'AUPRC': average_precision_score}


def safe(metric, y, p):
    return metric(y, p) if len(np.unique(y)) > 1 else np.nan


def cluster_bootstrap(y, preds, groups, metric):
    # Resample patients; every encoder is scored on the same resample.
    uq = np.unique(groups)
    by = {g: np.where(groups == g)[0] for g in uq}
    rng = np.random.default_rng(BOOT_SEED)
    out = {a: [] for a in preds}
    deltas = {a: [] for a in preds if a != REF}
    for _ in range(N_BOOT):
        idx = np.concatenate([by[g] for g in rng.choice(uq, size=len(uq), replace=True)])
        v = {a: safe(metric, y[idx], p[idx]) for a, p in preds.items()}
        for a in preds:
            if np.isfinite(v[a]):
                out[a].append(v[a])
            if a != REF and np.isfinite(v[a]) and np.isfinite(v[REF]):
                deltas[a].append(v[REF] - v[a])
    return out, deltas


summary = []
for name, o in OOF.items():
    df = TASKS[name].set_index('ACQUISITION_ID')
    o = o.sort_values(['rep', 'fold', 'id'])
    ref = o[o.arm == REF]
    y, groups = ref.label.values, df.loc[ref.id, 'pid'].values
    preds = {}
    for a in ARMS:
        oa = o[o.arm == a]
        assert (oa.id.values == ref.id.values).all()
        preds[a] = oa.prob.values
    for mname, metric in METRICS.items():
        boot, deltas = cluster_bootstrap(y, preds, groups, metric)
        for a in ARMS:
            row = dict(task=name, metric=mname, encoder=LABEL[a],
                       value=metric(y, preds[a]),
                       ci_lo=np.percentile(boot[a], 2.5), ci_hi=np.percentile(boot[a], 97.5))
            if a != REF:
                ds = np.asarray(deltas[a])
                row.update(delta_haiku_minus=metric(y, preds[REF]) - metric(y, preds[a]),
                           delta_lo=np.percentile(ds, 2.5), delta_hi=np.percentile(ds, 97.5),
                           p_vs_haiku=min(1.0, 2 * min((ds <= 0).mean(), (ds >= 0).mean())))
            summary.append(row)

summary = pd.DataFrame(summary)
summary.to_csv(OUTPUT_DIR / 'mil_cls_summary.csv', index=False)
pd.options.display.float_format = '{:.3f}'.format
summary

In [ ]:
# ---- figures: AUROC / AUPRC bars, pooled ROC and precision-recall curves ----
COLORS = {'haiku_postproj': '#2F5A8A', 'kronos': '#7FA7C9', 'eva': '#9BB89A',
          'virtues_official': '#C9A27F', 'virtues_ours': '#B5736B'}

for name, o in OOF.items():
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    s = summary[summary.task == name]
    x = np.arange(len(ARMS))
    for k, mname in enumerate(METRICS):
        sm = s[s.metric == mname].set_index('encoder').loc[[LABEL[a] for a in ARMS]]
        axes[0].bar(x + (k - 0.5) * 0.38, sm.value, 0.38, color=[COLORS[a] for a in ARMS],
                    alpha=1.0 if k == 0 else 0.55, edgecolor='black', linewidth=0.5,
                    yerr=[sm.value - sm.ci_lo, sm.ci_hi - sm.value], capsize=2, label=mname)
    axes[0].set_xticks(x); axes[0].set_xticklabels([LABEL[a] for a in ARMS], rotation=30, ha='right')
    axes[0].set_ylim(0, 1); axes[0].set_ylabel('Pooled out-of-fold score')
    axes[0].set_title('AUROC (solid) and AUPRC (light)')
    for a in ARMS:
        oa = o[o.arm == a]
        fpr, tpr, _ = roc_curve(oa.label, oa.prob)
        prec, rec, _ = precision_recall_curve(oa.label, oa.prob)
        axes[1].plot(fpr, tpr, color=COLORS[a], lw=1.5, label=LABEL[a])
        axes[2].plot(rec, prec, color=COLORS[a], lw=1.5, label=LABEL[a])
    axes[1].plot([0, 1], [0, 1], 'k--', lw=0.8)
    axes[2].axhline(o[o.arm == REF].label.mean(), color='k', ls='--', lw=0.8)
    axes[1].set_xlabel('False positive rate'); axes[1].set_ylabel('True positive rate')
    axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
    axes[1].set_title('ROC'); axes[2].set_title('Precision-recall')
    axes[2].legend(frameon=False, fontsize=8, loc='lower left')
    for ax in axes:
        ax.spines[['top', 'right']].set_visible(False)
    fig.suptitle(name, fontweight='bold')
    plt.tight_layout()
    for ext in ('svg', 'pdf', 'png'):
        fig.savefig(OUTPUT_DIR / f'mil_cls_{name}.{ext}', dpi=300, bbox_inches='tight')
    plt.show()